# Compute the LoRA Relationship Matrix

This notebook runs Step 14 of the thesis prototype. It computes a static relationship matrix $R$ from the geometry of two local GPT-2 LoRA adapters:

- `adapters/gpt2-helpful-adapter`
- `adapters/gpt2-harmless-adapter`

The thesis pipeline is:

$$\delta_i \rightarrow R \rightarrow \lambda = f(p, R) \rightarrow \theta(\lambda)$$

This notebook covers the $\delta_i \rightarrow R$ step by converting adapter parameters into a labeled cosine-similarity matrix.

## 1. Clone or update the repository

This cell always starts in `/content`. If `/content/master-thesis/.git` exists, it updates the repository. Otherwise, it clones a fresh copy. This avoids nested folders such as `/content/master-thesis/master-thesis`.

In [ ]:
%cd /content

from pathlib import Path
import subprocess

repo_path = Path("/content/master-thesis")

if (repo_path / ".git").is_dir():
    print("Repository already exists. Pulling the latest changes...")
    subprocess.run(["git", "-C", str(repo_path), "pull"], check=True)
else:
    print("Cloning the repository...")
    subprocess.run(
        ["git", "clone", "https://github.com/NZhang137/master-thesis.git"],
        check=True,
    )

%cd /content/master-thesis

## 2. Inspect the repository

You should see the repository root together with the scripts and source modules used for relationship-matrix computation.

In [ ]:
!pwd
!ls
!ls scripts
!ls src
!ls results || echo "No results folder found yet."

## 3. Install dependencies

The matrix computation uses PyTorch to flatten tensors and compute cosine similarity, `safetensors` to load PEFT adapter weights, and pandas for a convenient table preview. Working directly with adapter tensors keeps the computation lightweight.

In [ ]:
!pip install -q torch safetensors pandas

If you also want the general Hugging Face prototype dependencies in this session, you may optionally run:

```python
!pip install -q transformers datasets peft accelerate
```

The matrix script itself uses the smaller dependency set installed above.

## 4. Check whether the adapters exist

Both local adapter folders are required. If they already appear below, skip the upload step.

In [ ]:
from pathlib import Path

helpful_path = Path("adapters/gpt2-helpful-adapter")
harmless_path = Path("adapters/gpt2-harmless-adapter")

print("Helpful adapter exists:", helpful_path.is_dir())
print("Harmless adapter exists:", harmless_path.is_dir())
!ls adapters || echo "No adapters folder found."

## 5. Upload `adapters.zip` if needed

Run the next upload cell only when the adapter folders are missing. Select your local `adapters.zip` backup.

`adapters.zip` is a local backup only. It and the extracted adapter weights must not be committed to GitHub.

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
!if [ -f adapters.zip ]; then unzip -o adapters.zip; else echo "No adapters.zip found, skipping unzip."; fi
!ls adapters || echo "No adapters folder found."

## 6. Check the adapter files

The checker verifies that both folders contain `adapter_config.json` and a supported adapter-weight file.

In [ ]:
!python scripts/check_adapters.py

## 7. Compute the relationship matrix

The script loads the saved LoRA tensors, flattens them in a deterministic order, and computes pairwise cosine similarity. The resulting $2 \times 2$ matrix is a geometry proxy that requires empirical validation.

In [ ]:
!python scripts/compute_relationship_matrix.py

## 8. Inspect the outputs

The script creates two small files:

- `results/relationship_matrix.csv`
- `results/relationship_matrix_metadata.json`

The CSV contains the labeled matrix. The JSON records the adapter names, paths, representation, vector lengths, similarity type, and caveat.

In [ ]:
!cat results/relationship_matrix.csv
!cat results/relationship_matrix_metadata.json

In [ ]:
import pandas as pd

relationship_df = pd.read_csv(
    "results/relationship_matrix.csv",
    index_col="adapter",
)
relationship_df

## 9. Git safety check

It is okay to commit the small `relationship_matrix.csv` and metadata JSON files.

Do **not** commit:

- `adapters/`
- `adapters.zip`
- `.safetensors` or `.bin` files
- checkpoints or full model files

The adapter weights are generated local artifacts and should remain outside Git.

In [ ]:
!git status

## What this notebook establishes

This notebook computes the static relationship matrix $R$ from flattened LoRA adapter geometry for use in later coefficient-mapping experiments.